# Token Metrics Emitting lab

![flow](./img/token-metrics-emitting.gif)

Playground to try the [emit token metric policy](https://learn.microsoft.com/azure/api-management/azure-openai-emit-token-metric-policy). The policy sends metrics to Application Insights about consumption of large language model tokens through Azure AI Foundry APIs.

Notes:
- Token count metrics include: Total Tokens, Prompt Tokens, and Completion Tokens.
- This policy supports OpenAI response streaming! Use the [streaming tool](../../tools/streaming.ipynb) to test and troubleshoot response streaming.
- Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

[View policy configuration](policy.xml)

### ⚙️ Initialization

1. Get Az Cli authentication information
1. Get terraform outputs (prerequisite: run the [setup.ipynb](./setup.ipynb))

In [ ]:
import sys
sys.path.insert(1, './shared')  # add the shared directory to the Python path
import utils

terraform_dir = 'terraform'

output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

output = utils.run(f"terraform -chdir={terraform_dir} output -json", "Retrieved Terraform outputs", "Failed to retrieve Terraform outputs")

if output.success and output.json_data:
    apim_gateway_url = output.json_data['apim_gateway_url']['value']
    inference_api_path = output.json_data['token_metrics_emitting']['value']['inference_api_path']
    all_apis_key_1 = output.json_data['token_metrics_emitting']['value']['all_apis_key_1']
    all_apis_key_2 = output.json_data['token_metrics_emitting']['value']['all_apis_key_2']
    all_apis_key_3 = output.json_data['token_metrics_emitting']['value']['all_apis_key_3']
    inference_api_version = output.json_data['token_metrics_emitting']['value']['inference_api_version']
    deployment_model_name = output.json_data['token_metrics_emitting']['value']['deployment_model_name']
    resource_group_name = output.json_data['resource_group_name']['value']
    application_insights_name = output.json_data['application_insights_name']['value']

    utils.print_info(f"Resource Group Name: {resource_group_name}")
    utils.print_info(f"Application Insights Name: {application_insights_name}")
    utils.print_info(f"APIM Gateway URL: {apim_gateway_url}")
    utils.print_info(f"Inference API path: {inference_api_path}")
    utils.print_info(f"All API key 1: {all_apis_key_1}")
    utils.print_info(f"All API key 2: {all_apis_key_2}")
    utils.print_info(f"All API key 3: {all_apis_key_3}")
    utils.print_info(f"Inference API version: {inference_api_version}")
    utils.print_info(f"Deployment model name: {deployment_model_name}")

<a id='requests'></a>
### 🧪 Test the API using a direct HTTP call

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

In [ ]:
import json, requests, time

runs = 10
sleep_time_ms = 100
url = f"{apim_gateway_url}/{inference_api_path}/openai/deployments/{deployment_model_name}/chat/completions?api-version={inference_api_version}"
utils.print_info(f"Endpoint URL: {url}")
messages = {"messages": [
    {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
    {"role": "user", "content": "Can you tell me the time, please?"}
]}
api_runs = []

# Initialize a session for connection pooling and set any default headers
session = requests.Session()
session.headers.update({
    'api-key': all_apis_key_1,
    'x-user-id': 'alex'
})

try:
    for i in range(runs):
        print(f"▶️ Run {i+1}/{runs}:")

        start_time = time.time()
        response = session.post(url, json = messages)
        response_time = time.time() - start_time
        print(f"⌚ {response_time:.2f} seconds")

        utils.print_response_code(response)
        print(f"Response headers: {json.dumps(dict(response.headers), indent = 4)}")

        if "x-ms-region" in response.headers:
            print(f"x-ms-region: \x1b[1;32m{response.headers.get("x-ms-region")}\x1b[0m") # this header is useful to determine the region of the backend that served the request
            api_runs.append((response_time, response.headers.get("x-ms-region")))

        if (response.status_code == 200):
            data = json.loads(response.text)
            print(f"Token usage: {json.dumps(dict(data.get("usage")), indent = 4)}\n")
            print(f"💬 {data.get("choices")[0].get("message").get("content")}\n")
        else:
            print(f"{response.text}\n")

        time.sleep(sleep_time_ms/1000)
finally:
    # Close the session to release the connection
    session.close()


<a id='sdk'></a>
### 🧪 Execute multiple runs for each subscription using the Azure OpenAI Python SDK

We will send requests for each subscription. Adjust the `sleep_time_ms` and the number of `runs` to your test scenario.


In [ ]:
import time
from openai import AzureOpenAI

runs = 10
sleep_time_ms = 100

clients = [
    AzureOpenAI(
        azure_endpoint = f"{apim_gateway_url}/{inference_api_path}",
        api_key = all_apis_key_1,
        api_version = inference_api_version
    ),
    AzureOpenAI(
        azure_endpoint = f"{apim_gateway_url}/{inference_api_path}",
        api_key = all_apis_key_2,
        api_version = inference_api_version
    ),
    AzureOpenAI(
        azure_endpoint = f"{apim_gateway_url}/{inference_api_path}",
        api_key = all_apis_key_3,
        api_version = inference_api_version
    )
]

for i in range(runs):
    print(f"▶️ Run {i+1}/{runs}:")

    for j in range(0, 3):
        response = clients[j].chat.completions.create(
            model = deployment_model_name,
            messages = [
                {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
                {"role": "user", "content": "Can you tell me the time, please?"}
            ],
            extra_headers = {"x-user-id": "alex"}
        )
        print(f"💬 Subscription {j+1}: {response.choices[0].message.content}")

    print()

    time.sleep(sleep_time_ms/1000)


<a id='kql'></a>
### 🔍 Analyze Application Insights custom metrics with a KQL query

With this query you can get the custom metrics that were emitted by Azure APIM. Note that it may take a few minutes for data to become available.

In [ ]:
import pandas as pd

query = "\"" + "customMetrics \
| where name == 'Total Tokens' \
| extend parsedCustomDimensions = parse_json(customDimensions) \
| extend clientIP = tostring(parsedCustomDimensions.['Client IP']) \
| extend apiId = tostring(parsedCustomDimensions.['API ID']) \
| extend apimSubscription = tostring(parsedCustomDimensions.['Subscription ID']) \
| extend UserId = tostring(parsedCustomDimensions.['User ID']) \
| project timestamp, value, clientIP, apiId, apimSubscription, UserId \
| order by timestamp asc" + "\""

output = utils.run(f"az monitor app-insights query --app {application_insights_name} -g {resource_group_name} --analytics-query {query}",
    f"App Insights query succeeded", f"App Insights query  failed")

table = output.json_data['tables'][0]
df = pd.DataFrame(table.get("rows"), columns = [col.get("name") for col in table.get('columns')])
df['timestamp'] = pd.to_datetime(df['timestamp']).dt.strftime('%H:%M')

df


<a id='plot'></a>
### 🔍 Plot the custom metrics results

In [ ]:
# plot the results
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['figure.figsize'] = [15, 7]
if df.empty:
    print("No data to plot")
else:
    ax = df.plot(kind = 'line', x = 'timestamp', y = 'value', legend = False)
    plt.title('Total token usage over time')
    plt.xlabel('Time')
    plt.ylabel('Tokens')
    plt.show()

<a id='portal'></a>
### 🔍 See the metrics on the Azure Portal

Open the Application Insights resource, navigate to the Metrics blade, then select the defined namespace (openai). Choose the metric "Total Tokens" with a Sum aggregation. Then, apply splitting by 'Subscription Id' to view values for each dimension.

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.